# Can target identifications use fixation data instead of raw gaze?

`target_identifications.py` currently needs raw 600 Hz gaze samples to find the eye position at the
moment the subject pressed the key. If fixation data suffices, the module can move to stage 2 (operating
on persisted DataFrames rather than `Trial` objects).

We measure:
1. % of identifications that occur **during** a fixation
2. % of identifications that occur **immediately after** a fixation (within 50 ms)
3. % of identifications without an associated gaze sample (current data loss)
4. Agreement between fixation-based and gaze-based target assignment

In [ ]:
import pandas as pd
import numpy as np
import os
import config as cnfg
from data_models.LWSEnums import SignalDetectionCategoryEnum

In [ ]:
base = cnfg.OUTPUT_PATH
fixations = pd.read_pickle(os.path.join(base, 'fixations.pkl'))
idents = pd.read_pickle(os.path.join(base, 'idents.pkl'))
metadata = pd.read_pickle(os.path.join(base, 'metadata.pkl'))

# keep dominant eye only
dom = metadata.set_index(['subject', 'trial'])['dominant_eye']
fix_dom = fixations.set_index(['subject', 'trial']).index.map(dom)
fixations = fixations[fixations['eye'].to_numpy() == fix_dom.to_numpy()]

# only real identifications (not misses)
real_idents = idents[idents['identification_category'] != SignalDetectionCategoryEnum.MISS].copy()
print(f"Real identifications (non-miss): {len(real_idents)}")
print(f"Fixations (dominant eye): {len(fixations)}")

In [ ]:
JUST_AFTER_THRESHOLD_MS = 50

results = []
for _, ident_row in real_idents.iterrows():
    subj, trial, t = ident_row['subject'], ident_row['trial'], ident_row['time']
    trial_fixs = fixations[(fixations['subject'] == subj) & (fixations['trial'] == trial)]

    during = trial_fixs[(trial_fixs['start_time'] <= t) & (trial_fixs['end_time'] >= t)]
    just_after = trial_fixs[
        (trial_fixs['end_time'] < t) & (t - trial_fixs['end_time'] <= JUST_AFTER_THRESHOLD_MS)
    ]

    results.append({
        'subject': subj, 'trial': trial, 'time': t,
        'ident_target': ident_row['target'],
        'ident_category': ident_row['identification_category'],
        'ident_dist_dva': ident_row.get('distance_dva', np.nan),
        'during_fixation': len(during) > 0,
        'during_fix_target': during['target'].iloc[0] if len(during) > 0 else None,
        'just_after_fixation': len(just_after) > 0,
        'after_fix_target': just_after.iloc[-1]['target'] if len(just_after) > 0 else None,
        'any_nearby': len(during) > 0 or len(just_after) > 0,
    })

df = pd.DataFrame(results)

## 1. Identifications during a fixation

In [ ]:
n_during = df['during_fixation'].sum()
print(f"During a fixation: {n_during}/{len(df)} ({100*n_during/len(df):.1f}%)")

## 2. Identifications immediately after a fixation

In [ ]:
n_after = df['just_after_fixation'].sum()
n_either = df['any_nearby'].sum()
print(f"Within {JUST_AFTER_THRESHOLD_MS} ms after fixation end: {n_after}/{len(df)} ({100*n_after/len(df):.1f}%)")
print(f"Either during OR just after: {n_either}/{len(df)} ({100*n_either/len(df):.1f}%)")

## 3. Current data loss (identifications without associated gaze)

In [ ]:
n_no_gaze = real_idents[['left_x', 'right_x']].isna().all(axis=1).sum()
print(f"Without any gaze sample (current approach): {n_no_gaze}/{len(real_idents)} ({100*n_no_gaze/len(real_idents):.1f}%)")

## 4. Target assignment agreement

In [ ]:
during_only = df[df['during_fixation']].copy()
agree = (during_only['ident_target'] == during_only['during_fix_target']).sum()
print(f"Target agreement (ident during fixation, n={len(during_only)}):")
print(f"  Agree: {agree}/{len(during_only)} ({100*agree/len(during_only):.1f}%)")

after_only = df[~df['during_fixation'] & df['just_after_fixation']].copy()
if len(after_only) > 0:
    agree_after = (after_only['ident_target'] == after_only['after_fix_target']).sum()
    print(f"  Just-after agree: {agree_after}/{len(after_only)} ({100*agree_after/len(after_only):.1f}%)")

## Disagreement and edge case analysis

In [ ]:
# disagreements: fixation-based target != gaze-based target
during_only = df[df['during_fixation']].copy()
disagree = during_only[during_only['ident_target'] != during_only['during_fix_target']]
print(f"Disagreements: {len(disagree)}")
for _, row in disagree.iterrows():
    print(f"  S{row['subject']} T{row['trial']}: gaze->{row['ident_target']}, fix->{row['during_fix_target']}")
    print(f"    category={row['ident_category']}, gaze_dist_dva={row['ident_dist_dva']:.2f}")

# identifications with no nearby fixation
neither = df[~df['any_nearby']]
print(f"\nNo nearby fixation: {len(neither)}")
for _, row in neither.iterrows():
    subj, trial, t = row['subject'], row['trial'], row['time']
    trial_fixs = fixations[(fixations['subject'] == subj) & (fixations['trial'] == trial)]
    time_diffs = (trial_fixs['end_time'] - t).abs()
    nearest = trial_fixs.loc[time_diffs.idxmin()]
    gap = t - nearest['end_time']
    print(f"  S{subj} T{trial}: t={t:.0f}, nearest fix ends at {nearest['end_time']:.0f} (gap={gap:.0f} ms)")

## Conclusion

**Results (2026-08-11, on built pickles with the old `target{j}` naming):**

| Metric | Value |
| --- | --- |
| Identifications during a fixation | 1223/1248 (98.0%) |
| During OR within 50 ms after | 1246/1248 (99.8%) |
| Current approach data loss | 0/1248 (0.0%) |
| Target agreement (during fixation) | 1222/1223 (99.9%) |
| Target agreement (just after) | 23/23 (100.0%) |

The single disagreement is a **false alarm** where two targets are nearly equidistant (6.17 vs 6.20 DVA),
both well above the 1.75 DVA on-target threshold. The fixation picked one, the raw gaze picked the other.
Classification (false alarm) is identical either way.

The 2 "neither" cases (0.2%) are identifications during saccades, 117 ms and 156 ms from the nearest
fixation. These would be dropped by a fixation-based approach. Both are hits in the current system.

**Decision: fixation data is sufficient for target identification.** `target_identifications` can move
to stage 2, using the fixation that contains (or immediately precedes) the identification action.
The 0.2% loss of saccade-time identifications is acceptable.